# 2층 신경망의 순전파와 역전파

입력 2개, 은닉 2개, 출력 1개인 신경망의 순전파를 적고, 출력층에서 입력층 방향으로 연쇄 법칙을 적용해 기울기를 유도한다. 이어서 고정된 숫자 예제를 손계산하고, 같은 식을 NumPy로 다시 계산해 소수점 넷째 자리까지 비교한다.

자동미분 라이브러리는 쓰지 않는다. 활성화는 시그모이드, 손실은 이진 교차 엔트로피(BCE)이다.

과제 문장에 정답 라벨의 숫자가 비어 있다. 이 노트북은 **y_true = 1** 로 고정한다.


## 1. 구조와 순전파

열벡터 규약을 쓴다. 은닉층 가중치 \(W^{(1)}\)의 각 행이 은닉 유닛 하나이고, 출력층 가중치 \(W^{(2)}\)는 길이가 2인 벡터이다.

| 기호 | shape | 의미 |
| --- | --- | --- |
| \(x\) | (2,) | 입력 |
| \(W^{(1)}\) | (2, 2) | 입력 → 은닉 |
| \(b^{(1)}\) | (2,) | 은닉 편향 |
| \(z^{(1)}\) | (2,) | 은닉 로짓 |
| \(a^{(1)}\) | (2,) | 은닉 활성화 |
| \(W^{(2)}\) | (2,) | 은닉 → 출력 |
| \(b^{(2)}\) | () | 출력 편향 |
| \(z^{(2)}\) | () | 출력 로짓 |
| \(\hat{y}\) | () | 예측 확률 |
| \(y\) | () | 정답 0 또는 1 |

시그모이드와 손실은 다음과 같다.

$$
\sigma(z)=\frac{1}{1+e^{-z}},\qquad
\sigma'(z)=\sigma(z)\,(1-\sigma(z))
$$

$$
L=\big(y\log\hat{y}+(1-y)\log(1-\hat{y})\big)\text{의 부호를 바꾼 값}
= -\big(y\log\hat{y}+(1-y)\log(1-\hat{y})\big)
$$

순전파는 네 단계다.

$$
z^{(1)}=W^{(1)} x+b^{(1)}
$$

$$
a^{(1)}=\sigma\big(z^{(1)}\big)
$$

$$
z^{(2)}=W^{(2)}\cdot a^{(1)}+b^{(2)}
$$

$$
\hat{y}=\sigma\big(z^{(2)}\big)
$$

\(W^{(1)} x\)는 행렬-벡터 곱이라 shape이 (2, 2) × (2,) = (2,)이다. \(W^{(2)}\cdot a^{(1)}\)는 내적이라 스칼라가 된다.


## 2. 출력층에서 입력층으로의 역전파

각 단계에서 연쇄 법칙과 텐서 shape을 같이 적는다.

### 2.1 \(\partial L/\partial\hat{y}\)  — shape ()

\(y\in\{0,1\}\)이고 \(\hat{y}\in(0,1)\)일 때

$$
\frac{\partial L}{\partial\hat{y}}
= -\frac{y}{\hat{y}}+\frac{1-y}{1-\hat{y}}.
$$

\(y=1\)이면 \(-1/\hat{y}\), \(y=0\)이면 \(1/(1-\hat{y})\)이다.

### 2.2 \(\partial L/\partial z^{(2)}\)  — shape ()

$$
\frac{\partial\hat{y}}{\partial z^{(2)}}=\hat{y}(1-\hat{y})
$$

$$
\frac{\partial L}{\partial z^{(2)}}
=\frac{\partial L}{\partial\hat{y}}\,\hat{y}(1-\hat{y})
=\left(-\frac{y}{\hat{y}}+\frac{1-y}{1-\hat{y}}\right)\hat{y}(1-\hat{y}).
$$

괄호를 풀면

$$
- y(1-\hat{y})+(1-y)\hat{y}
= -y+y\hat{y}+\hat{y}-y\hat{y}
= \hat{y}-y.
$$

따라서 BCE와 시그모이드를 이으면 \(\partial L/\partial z^{(2)}=\hat{y}-y\)이다. 아래 손계산에서는 나누기 전의 식과 이 단순화 결과가 같은지 둘 다 적는다.

### 2.3 \(\partial L/\partial W^{(2)}\)  — shape (2,)

\(z^{(2)}=W^{(2)}\cdot a^{(1)}+b^{(2)}\)이므로 \(W^{(2)}\)에 대한 기울기는 스칼라 기울기에 활성화 벡터를 곱한 것이다.

$$
\frac{\partial L}{\partial W^{(2)}}
=\frac{\partial L}{\partial z^{(2)}}\, a^{(1)}
\qquad
()\times(2,)=(2,)
$$

편향은 \(\partial L/\partial b^{(2)}=\partial L/\partial z^{(2)}\)이고 shape은 ()이다. 과제 필수 목록에는 없지만 순전파에 들어 있으므로 함께 적는다.

### 2.4 \(\partial L/\partial a^{(1)}\)  — shape (2,)

같은 내적을 \(a^{(1)}\) 쪽으로 보면 가중치가 남는다.

$$
\frac{\partial L}{\partial a^{(1)}}
=\frac{\partial L}{\partial z^{(2)}}\, W^{(2)}
\qquad
()\times(2,)=(2,)
$$

### 2.5 \(\partial L/\partial z^{(1)}\)  — shape (2,)

은닉 활성화는 성분마다 시그모이드이다. 야코비안은 대각행렬 \(\mathrm{diag}\big(a^{(1)}\odot(1-a^{(1)})\big)\)이므로 벡터 기울기는 원소별 곱이다.

$$
\frac{\partial L}{\partial z^{(1)}}
=\frac{\partial L}{\partial a^{(1)}}
\odot a^{(1)}\odot\big(1-a^{(1)}\big)
\qquad (2,)
$$

\(\partial L/\partial b^{(1)}\)도 이 벡터와 같다.

### 2.6 \(\partial L/\partial W^{(1)}\)  — shape (2, 2)

\(z^{(1)}=W^{(1)} x+b^{(1)}\)에서 \(z^{(1)}\)의 기울기를 \(g\)라고 하면

$$
\frac{\partial L}{\partial W^{(1)}}=g\, x^{\top}
=\frac{\partial L}{\partial z^{(1)}}\, x^{\top}.
$$

shape은 (2, 1) × (1, 2) = (2, 2)이다. \(x=(1,0)\)이면 첫 번째 열만 남고 두 번째 열은 0이다. 입력이 0인 가중치로는 이번 샘플의 손실이 바뀌지 않기 때문이다.


## 3. 고정 예제의 손계산

### 입력 체크포인트

| 텐서 | shape | 값 |
| --- | --- | --- |
| \(x\) | (2,) | \([1,\ 0]\) |
| \(W^{(1)}\) | (2, 2) | \([[0.1,\ 0.2],\ [0.3,\ 0.4]]\) |
| \(b^{(1)}\) | (2,) | \([0,\ 0]\) |
| \(W^{(2)}\) | (2,) | \([0.5,\ 0.6]\) |
| \(b^{(2)}\) | () | \(0\) |
| \(y\) | () | \(1\) |

### 순전파 체크포인트

**\(z^{(1)}\), shape (2,)**

$$
z^{(1)}_1=0.1\cdot 1+0.2\cdot 0+0=0.1
$$

$$
z^{(1)}_2=0.3\cdot 1+0.4\cdot 0+0=0.3
$$

$$
z^{(1)}=[0.1,\ 0.3]
$$

**\(a^{(1)}\), shape (2,)**

$$
e^{-0.1}=0.904837418,\quad
\sigma(0.1)=\frac{1}{1.904837418}=0.524979187
$$

$$
e^{-0.3}=0.740818221,\quad
\sigma(0.3)=\frac{1}{1.740818221}=0.574442517
$$

$$
a^{(1)}=[0.524979187,\ 0.574442517]
$$

**\(z^{(2)}\), shape ()**

$$
z^{(2)}=0.5\cdot 0.524979187+0.6\cdot 0.574442517+0
$$

$$
=0.262489594+0.344665510=0.607155104
$$

**\(\hat{y}\), shape ()**

$$
e^{-0.607155104}=0.544947\text{ 부근},\quad
\hat{y}=\sigma(0.607155104)=0.647291570
$$

정답이 1이므로 손실은 \(L=-\log\hat{y}=0.434958437\)이다. 필수 체크포인트는 \(z^{(1)}, a^{(1)}, z^{(2)}, \hat{y}\) 네 개다.

### 역전파 체크포인트

**\(\partial L/\partial\hat{y}\), shape ()**

$$
\frac{\partial L}{\partial\hat{y}}=-\frac{1}{0.647291570}=-1.544898847
$$

**\(\partial L/\partial z^{(2)}\), shape ()**

$$
\hat{y}(1-\hat{y})=0.647291570\cdot 0.352708430=0.228308
$$

$$
\frac{\partial L}{\partial\hat{y}}\,\hat{y}(1-\hat{y})
=-1.544898847\cdot 0.647291570\cdot 0.352708430
=-0.352708430
$$

단순화 \(\hat{y}-y=0.647291570-1=-0.352708430\)과 같다.

**\(\partial L/\partial W^{(2)}\), shape (2,)**

$$
(-0.352708430)\,[0.524979187,\ 0.574442517]
=[-0.185164585,\ -0.202610718]
$$

**\(\partial L/\partial a^{(1)}\), shape (2,)**

$$
(-0.352708430)\,[0.5,\ 0.6]
=[-0.176354215,\ -0.211625058]
$$

**\(\partial L/\partial z^{(1)}\), shape (2,)**

시그모이드 도함수:

$$
a^{(1)}\odot(1-a^{(1)})
=[0.249376040,\ 0.244458312]
$$

$$
\frac{\partial L}{\partial z^{(1)}}
=[-0.176354215\cdot 0.249376040,\ 
-0.211625058\cdot 0.244458312]
=[-0.043978516,\ -0.051733504]
$$

**\(\partial L/\partial W^{(1)}\), shape (2, 2)**

\(x=(1,0)\)과의 외적이다. 두 번째 열은 0이다.

$$
\frac{\partial L}{\partial W^{(1)}}
=
\begin{bmatrix}
-0.043978516 & 0\\
-0.051733504 & 0
\end{bmatrix}
$$

### 소수점 넷째 자리 기준값

비교에 쓰는 값은 위에서 구한 수를 소수 넷째 자리에서 반올림한 결과이다.

| 텐서 | shape | 반올림 값 |
| --- | --- | --- |
| \(z^{(1)}\) | (2,) | [0.1000, 0.3000] |
| \(a^{(1)}\) | (2,) | [0.5250, 0.5744] |
| \(z^{(2)}\) | () | 0.6072 |
| \(\hat{y}\) | () | 0.6473 |
| \(\partial L/\partial\hat{y}\) | () | -1.5449 |
| \(\partial L/\partial z^{(2)}\) | () | -0.3527 |
| \(\partial L/\partial W^{(2)}\) | (2,) | [-0.1852, -0.2026] |
| \(\partial L/\partial a^{(1)}\) | (2,) | [-0.1764, -0.2116] |
| \(\partial L/\partial z^{(1)}\) | (2,) | [-0.0440, -0.0517] |
| \(\partial L/\partial W^{(1)}\) | (2, 2) | [[-0.0440, 0.0000], [-0.0517, 0.0000]] |


In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
np.random.seed(42)


def sigmoid(z):
    """시그모이드 σ(z) = 1 / (1 + exp(-z))."""
    return 1.0 / (1.0 + np.exp(-z))


# shape: x (2,), W1 (2, 2), b1 (2,), W2 (2,), b2 (), y ()
x = np.array([1.0, 0.0])
W1 = np.array([[0.1, 0.2], [0.3, 0.4]])
b1 = np.array([0.0, 0.0])
W2 = np.array([0.5, 0.6])
b2 = 0.0
y_true = 1.0

# 순전파. z1 = W1 @ x + b1
z1 = W1 @ x + b1          # (2,)
a1 = sigmoid(z1)          # (2,)
z2 = float(W2 @ a1 + b2)  # ()
y_pred = float(sigmoid(z2))  # ()

# 역전파. BCE 미분 후 시그모이드 도함수를 곱한다.
dL_dy = -y_true / y_pred + (1.0 - y_true) / (1.0 - y_pred)  # ()
dL_dz2 = dL_dy * y_pred * (1.0 - y_pred)  # ()
dL_dz2_simplified = y_pred - y_true       # () , ŷ - y
dL_dW2 = dL_dz2 * a1                      # (2,)
dL_da1 = dL_dz2 * W2                      # (2,)
dL_dz1 = dL_da1 * a1 * (1.0 - a1)         # (2,)
dL_dW1 = np.outer(dL_dz1, x)              # (2, 2)

hand = {
    "z1": np.array([0.1000, 0.3000]),
    "a1": np.array([0.5250, 0.5744]),
    "z2": np.array(0.6072),
    "y_pred": np.array(0.6473),
    "dL_dy": np.array(-1.5449),
    "dL_dz2": np.array(-0.3527),
    "dL_dW2": np.array([-0.1852, -0.2026]),
    "dL_da1": np.array([-0.1764, -0.2116]),
    "dL_dz1": np.array([-0.0440, -0.0517]),
    "dL_dW1": np.array([[-0.0440, 0.0000], [-0.0517, 0.0000]]),
}
computed = {
    "z1": z1,
    "a1": a1,
    "z2": np.array(z2),
    "y_pred": np.array(y_pred),
    "dL_dy": np.array(dL_dy),
    "dL_dz2": np.array(dL_dz2),
    "dL_dW2": dL_dW2,
    "dL_da1": dL_da1,
    "dL_dz1": dL_dz1,
    "dL_dW1": dL_dW1,
}
shapes = {
    "x": x.shape,
    "W1": W1.shape,
    "b1": b1.shape,
    "z1": z1.shape,
    "a1": a1.shape,
    "W2": W2.shape,
    "z2": (),
    "y_pred": (),
    "dL_dy": (),
    "dL_dz2": (),
    "dL_dW2": dL_dW2.shape,
    "dL_da1": dL_da1.shape,
    "dL_dz1": dL_dz1.shape,
    "dL_dW1": dL_dW1.shape,
}

print("shape")
for name, shape in shapes.items():
    print(f"  {name:8s} {shape}")
print()
print(f"∂L/∂z2 직접 계산 {dL_dz2:.10f}")
print(f"∂L/∂z2 = y_pred - y {dL_dz2_simplified:.10f}")
print()
print(f"{'이름':12s} {'NumPy 원값':22s} {'소수 4자리':22s} {'손계산':22s}")
for name, expected in hand.items():
    raw = computed[name]
    got = np.round(raw, 4)
    assert np.array_equal(got, expected), (name, got, expected)
    print(f"{name:12s} {np.array2string(raw, precision=6):22s} {np.array2string(got, precision=4):22s} {np.array2string(expected, precision=4):22s}")
print()
print("소수점 4자리까지 손계산과 일치한다.")


shape
  x        (2,)
  W1       (2, 2)
  b1       (2,)
  z1       (2,)
  a1       (2,)
  W2       (2,)
  z2       ()
  y_pred   ()
  dL_dy    ()
  dL_dz2   ()
  dL_dW2   (2,)
  dL_da1   (2,)
  dL_dz1   (2,)
  dL_dW1   (2, 2)

∂L/∂z2 직접 계산 -0.3527084300
∂L/∂z2 = y_pred - y -0.3527084300

이름           NumPy 원값               소수 4자리                 손계산                   
z1           [0.1 0.3]              [0.1 0.3]              [0.1 0.3]             
a1           [0.524979 0.574443]    [0.525  0.5744]        [0.525  0.5744]       
z2           0.607155               0.6072                 0.6072                
y_pred       0.647292               0.6473                 0.6473                
dL_dy        -1.544899              -1.5449                -1.5449               
dL_dz2       -0.352708              -0.3527                -0.3527               
dL_dW2       [-0.185165 -0.202611]  [-0.1852 -0.2026]      [-0.1852 -0.2026]     
dL_da1       [-0.176354 -0.211625]  [-0.1764 -0.2116]  